# 📊 02 — Exploratory Data Analysis
**E-Commerce Customer & Revenue Analytics Platform**

This notebook generates 7 business-critical EDA charts:
1. Monthly Revenue Trend
2. Top 20 Products by Revenue
3. Revenue by Country
4. Customer Purchase Frequency Distribution
5. Order Revenue Distribution Histogram
6. Feature Correlation Heatmap
7. Month-over-Month Revenue Growth Rate

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

CLEAN_PATH = os.path.join('..', 'data', 'processed', 'cleaned_data.csv')
df = pd.read_csv(CLEAN_PATH, parse_dates=['InvoiceDate'])
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols')
df.head(3)

## Chart 1: Monthly Revenue Trend

In [ ]:
monthly = df.groupby('YearMonth')['Revenue'].sum().reset_index().sort_values('YearMonth')
monthly['MA3'] = monthly['Revenue'].rolling(3, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fa')

ax.plot(monthly['YearMonth'], monthly['Revenue'],
        color='#6C63FF', linewidth=2.4, label='Monthly Revenue', zorder=3)
ax.fill_between(monthly['YearMonth'], monthly['Revenue'], alpha=0.12, color='#6C63FF')
ax.plot(monthly['YearMonth'], monthly['MA3'],
        '--', color='#FFB347', linewidth=1.8, alpha=0.85, label='3-Month Moving Avg')

ax.set_title('Monthly Revenue Trend (2022–2023)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.yaxis.grid(True, color='#ddd', linewidth=0.8)
ax.set_axisbelow(True)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', '01_monthly_revenue_trend.png'), dpi=150, bbox_inches='tight')
plt.show()

## Chart 2: Top 20 Products by Revenue

In [ ]:
top20 = (df.groupby('Description')['Revenue']
           .sum().reset_index()
           .sort_values('Revenue', ascending=False)
           .head(20))

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fa')
colors = ['#6C63FF' if i < 5 else '#43B6C8' for i in range(len(top20))]
ax.barh(top20['Description'], top20['Revenue'], color=colors, edgecolor='white', height=0.7)
ax.set_title('Top 20 Products by Revenue', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Revenue (£)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.yaxis.grid(False)
ax.xaxis.grid(True, color='#ddd', linewidth=0.8)
ax.set_axisbelow(True)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', '02_top_20_products.png'), dpi=150, bbox_inches='tight')
plt.show()

## Chart 3: Revenue by Country

In [ ]:
country = (df.groupby('Country')['Revenue']
             .sum().reset_index()
             .sort_values('Revenue', ascending=False)
             .head(15))

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fa')
palette = ['#6C63FF'] + ['#43B6C8'] * (len(country) - 1)
ax.bar(country['Country'], country['Revenue'], color=palette, edgecolor='white', width=0.65)
ax.set_title('Revenue by Country (Top 15)', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Revenue (£)')
ax.tick_params(axis='x', rotation=35)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.yaxis.grid(True, color='#ddd', linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', '03_revenue_by_country.png'), dpi=150, bbox_inches='tight')
plt.show()

## Chart 4: Customer Purchase Frequency Distribution

In [ ]:
order_counts = df.groupby('CustomerID')['InvoiceNo'].nunique()
capped = order_counts.clip(upper=30)

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fa')
ax.hist(capped, bins=30, color='#6C63FF', edgecolor='white', alpha=0.85)
ax.axvline(capped.mean(), color='#FF6B6B', linewidth=2, linestyle='--',
           label=f'Mean: {capped.mean():.1f} orders')
ax.axvline(capped.median(), color='#FFB347', linewidth=2, linestyle=':',
           label=f'Median: {int(capped.median())} orders')
ax.set_title('Customer Purchase Frequency Distribution', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Orders (capped at 30)')
ax.set_ylabel('Number of Customers')
ax.yaxis.grid(True, color='#ddd', linewidth=0.8)
ax.set_axisbelow(True)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', '04_customer_purchase_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## Chart 5: Revenue Distribution Histogram

In [ ]:
order_rev = df.groupby('InvoiceNo')['Revenue'].sum()
capped_rev = order_rev.clip(upper=order_rev.quantile(0.99))

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fa')
ax.hist(capped_rev, bins=50, color='#43B6C8', edgecolor='white', alpha=0.85)
ax.axvline(capped_rev.mean(), color='#FF6B6B', linestyle='--', linewidth=2,
           label=f'Mean: £{capped_rev.mean():,.2f}')
ax.set_title('Order Revenue Distribution (99th pct capped)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Order Revenue (£)')
ax.set_ylabel('Number of Orders')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.yaxis.grid(True, color='#ddd', linewidth=0.8)
ax.set_axisbelow(True)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', '05_revenue_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## Chart 6: Correlation Heatmap

In [ ]:
import numpy as np
num_cols = ['Quantity', 'UnitPrice', 'Revenue', 'OrderYear', 'OrderMonth', 'OrderDay']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('white')
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, linecolor='white',
            annot_kws={'size': 11}, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', '06_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## Chart 7: Month-over-Month Revenue Growth Rate

In [ ]:
monthly = df.groupby('YearMonth')['Revenue'].sum().reset_index().sort_values('YearMonth')
monthly['Growth'] = monthly['Revenue'].pct_change() * 100
monthly = monthly.dropna(subset=['Growth'])

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fa')
colors = ['#56D79E' if g >= 0 else '#FF6B6B' for g in monthly['Growth']]
ax.bar(monthly['YearMonth'], monthly['Growth'], color=colors, edgecolor='white', width=0.7)
ax.axhline(0, color='#555', linewidth=1.2)
ax.set_title('Month-over-Month Revenue Growth Rate (%)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month')
ax.set_ylabel('Growth Rate (%)')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.grid(True, color='#ddd', linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', '07_revenue_growth_rate.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Average MoM Growth: {monthly["Growth"].mean():.1f}%')

## ✅ Summary

All 7 EDA charts saved to `visualizations/`.

**Next**: `03_customer_segmentation.ipynb`